# Avaliação do Agente — 50 perguntas por edital

Roda 50 perguntas em **memória off** (cada pergunta independente), captura tokens
de entrada/saída por turno e salva em XLSX. No fim, extrapola o custo para os
demais modelos pagos a partir dos tokens medidos.

**Antes de rodar:**
- Coloca este notebook em `evals/` (na raiz do projeto).
- Coloca os 3 xlsx de perguntas em `evals/perguntas/`.
- Garante que o `.env` da raiz tem `OPENAI_API_KEY`.


In [1]:
import os, sys, time
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from agent.agent import build_agent, ask
from langchain_community.callbacks import get_openai_callback

print(f"cwd:  {Path.cwd()}")
print(f"root: {ROOT}")

cwd:  /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/evals
root: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant


## Configuração

In [2]:
EDITAIS = {
    "bndes":     1,
    "cvm":       2,
    "petrobras": 3,
}

EDITAL_NOME = "cvm"           # bndes | cvm | petrobras
PROVIDER    = "openai"
MODEL       = "gpt-4o-mini"

EDITAL_ID = EDITAIS[EDITAL_NOME]
ARQ_PERGUNTAS = Path("perguntas") / f"{EDITAL_NOME}.xlsx"

print(f"Edital:    {EDITAL_NOME} (id={EDITAL_ID})")
print(f"Modelo:    {PROVIDER}/{MODEL}")
print(f"Perguntas: {ARQ_PERGUNTAS}")

Edital:    cvm (id=2)
Modelo:    openai/gpt-4o-mini
Perguntas: perguntas/cvm.xlsx


## Carregar perguntas

In [3]:
df_q = pd.read_excel(ARQ_PERGUNTAS)
df_q = df_q.dropna(subset=["pergunta"])
df_q = df_q[df_q["pergunta"].str.strip() != ""].reset_index(drop=True)
print(f"{len(df_q)} perguntas carregadas")
df_q.head()

50 perguntas carregadas


,id,categoria,pergunta
0,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso CVM?
1,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...
2,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...
3,4,Dados sobre as inscrições,Quais são as formas de pagamento da taxa de in...
4,5,Dados sobre as inscrições,Qual é o prazo-limite de pagamento da inscriçã...


## Construir o agente

In [4]:
agente = build_agent(provider=PROVIDER, model=MODEL)
print(f"Agente: {PROVIDER}/{MODEL}")

Agente: openai/gpt-4o-mini


## Rodar as 50 perguntas

`chat_history=[]` em todas as chamadas → memória off. `get_openai_callback()`
captura tokens de **todas** as chamadas LLM dentro do escopo (invocação inicial
+ iterações de tool-call + eventual self-check).

In [5]:
resultados = []

for i, row in df_q.iterrows():
    pergunta = row["pergunta"]
    t0 = time.time()
    erro = None
    resposta = None

    try:
        with get_openai_callback() as cb:
            resposta = ask(
                agent=agente,
                question=pergunta,
                chat_history=[],          # memória OFF
                edital_id_ativo=EDITAL_ID,
            )
        in_tok    = cb.prompt_tokens
        out_tok   = cb.completion_tokens
        custo_usd = cb.total_cost
    except Exception as e:
        erro = str(e)
        in_tok = out_tok = 0
        custo_usd = 0.0

    latencia = round(time.time() - t0, 2)

    resultados.append({
        "id":            row["id"],
        "categoria":     row.get("categoria", ""),
        "pergunta":      pergunta,
        "resposta":      resposta,
        "input_tokens":  in_tok,
        "output_tokens": out_tok,
        "total_tokens":  in_tok + out_tok,
        "custo_usd":     round(custo_usd, 6),
        "latencia_s":    latencia,
        "erro":          erro,
    })

    flag = "ERRO" if erro else "ok"
    print(f"[{i+1:>2}/{len(df_q)}] {flag:>4}  {latencia:>5}s  "
          f"in={in_tok:>5}  out={out_tok:>4}")

df_r = pd.DataFrame(resultados)
print(f"\n{len(df_r)} respostas coletadas")

Carregando BGE-M3 (primeira vez pode demorar)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 50693.11it/s]


[ 1/50]   ok  11.27s  in= 7794  out= 115
[ 2/50]   ok   3.99s  in= 6123  out=  67
[ 3/50]   ok   3.68s  in= 6651  out= 113
[ 4/50]   ok   7.58s  in= 5979  out= 217
[ 5/50]   ok   5.02s  in= 6133  out= 160
[ 6/50]   ok   5.43s  in= 6053  out= 203
[ 7/50]   ok   4.71s  in= 6059  out= 101
[ 8/50]   ok   4.81s  in= 6200  out= 110
[ 9/50]   ok   3.28s  in= 6769  out= 122
[10/50]   ok   5.12s  in= 5952  out= 113
[11/50]   ok   3.99s  in= 6361  out= 105
[12/50]   ok   7.78s  in=12823  out= 177
[13/50]   ok   8.19s  in= 9856  out= 162
[14/50]   ok  16.69s  in=16240  out= 210
[15/50]   ok   7.67s  in=14612  out= 274
[16/50]   ok   9.02s  in=16415  out= 299
[17/50]   ok    4.5s  in= 6151  out= 114
[18/50]   ok   5.22s  in= 7756  out= 246
[19/50]   ok   5.12s  in= 7751  out= 129
[20/50]   ok   9.01s  in= 7578  out=  87
[21/50]   ok   5.73s  in= 6396  out= 211
[22/50]   ok   5.02s  in= 6277  out= 200
[23/50]   ok   3.99s  in= 6245  out= 107
[24/50]   ok   2.87s  in= 6407  out=  91
[25/50]   ok   9

## Salvar

In [6]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
slug_modelo = MODEL.replace("/", "_").replace(":", "_")
nome_base = f"{EDITAL_NOME}__{PROVIDER}__{slug_modelo}__{ts}"

Path("resultados").mkdir(exist_ok=True)
arq = Path("resultados") / f"{nome_base}.xlsx"
df_r.to_excel(arq, index=False)
print(f"Salvo: {arq}")

Salvo: resultados/cvm__openai__gpt-4o-mini__20260429_055441.xlsx


## Sumário

In [7]:
total_in   = int(df_r["input_tokens"].sum())
total_out  = int(df_r["output_tokens"].sum())
total_cost = float(df_r["custo_usd"].sum())
n_erros    = int(df_r["erro"].notna().sum())
lat_media  = float(df_r["latencia_s"].mean())
lat_p95    = float(df_r["latencia_s"].quantile(0.95))

print(f"Perguntas:        {len(df_r)}  ({n_erros} com erro)")
print(f"Tokens entrada:   {total_in:,}")
print(f"Tokens saída:     {total_out:,}")
print(f"Tokens totais:    {total_in + total_out:,}")
print(f"Custo medido:     US$ {total_cost:.4f}   ({PROVIDER}/{MODEL})")
print(f"Latência média:   {lat_media:.1f}s")
print(f"Latência p95:     {lat_p95:.1f}s")

Perguntas:        50  (0 com erro)
Tokens entrada:   427,696
Tokens saída:     8,665
Tokens totais:    436,361
Custo medido:     US$ 0.0644   (openai/gpt-4o-mini)
Latência média:   6.0s
Latência p95:     10.3s


## Estimativa de custo para os outros modelos

Multiplica os tokens medidos pelo preço de cada modelo.

> **Caveats:**
> - Tokenizers diferem entre modelos. Claude 4.7, por exemplo, usa tokenizer novo
>   que conta ~35% mais tokens para o mesmo texto. Estimativa = ordem de grandeza.
> - Modelos com raciocínio explícito (Opus, GPT-5.5) podem produzir mais tokens de
>   saída que o gpt-4o-mini no mesmo turno; a estimativa de output é piso.
> - Preços marcados `APROX` precisam ser conferidos antes de decidir orçamento.
> - Coluna `3 editais` assume volume similar entre os três PDFs (BNDES é o menor;
>   CVM é o maior — então o número é uma média grosseira).

In [8]:
# preços em USD por 1M tokens — (input, output)
PRECOS = {
    "openai/gpt-4o-mini":             (0.15,  0.60),   # confirmado
    "openai/gpt-5.4-mini":            (0.25,  2.00),   # APROX -- verificar
    "openai/gpt-5.4":                 (1.25, 10.00),   # APROX -- verificar
    "openai/gpt-5.5":                 (5.00, 30.00),   # confirmado
    "google/gemini-2.5-flash-lite":   (0.10,  0.40),   # confirmado
    "google/gemini-3-flash-preview":  (0.30,  2.50),   # APROX -- verificar
    "google/gemini-3.1-pro-preview":  (1.50, 10.00),   # APROX (base <200k) -- verificar
    "anthropic/claude-haiku-4-5":     (1.00,  5.00),   # confirmado
    "anthropic/claude-sonnet-4-6":    (3.00, 15.00),   # confirmado
    "anthropic/claude-opus-4-7":      (5.00, 25.00),   # confirmado
}

linhas = []
for nome, (pi, po) in PRECOS.items():
    custo_1ed = total_in / 1_000_000 * pi + total_out / 1_000_000 * po
    linhas.append({
        "modelo":                nome,
        "preco_in_usd_p_1M":     pi,
        "preco_out_usd_p_1M":    po,
        "custo_50q_1edital":     round(custo_1ed, 4),
        "custo_50q_3editais":    round(custo_1ed * 3, 4),
    })

df_custos = (
    pd.DataFrame(linhas)
    .sort_values("custo_50q_1edital")
    .reset_index(drop=True)
)
df_custos

,modelo,preco_in_usd_p_1M,preco_out_usd_p_1M,custo_50q_1edital,custo_50q_3editais
0,google/gemini-2.5-flash-lite,0.10,0.4,0.0462,0.1387
1,openai/gpt-4o-mini,0.15,0.6,0.0694,0.2081
2,openai/gpt-5.4-mini,0.25,2.0,0.1243,0.3728
3,google/gemini-3-flash-preview,0.30,2.5,0.1500,0.4499
4,anthropic/claude-haiku-4-5,1.00,5.0,0.4710,1.4131
5,openai/gpt-5.4,1.25,10.0,0.6213,1.8638
6,google/gemini-3.1-pro-preview,1.50,10.0,0.7282,2.1846
7,anthropic/claude-sonnet-4-6,3.00,15.0,1.4131,4.2392
8,anthropic/claude-opus-4-7,5.00,25.0,2.3551,7.0653
9,openai/gpt-5.5,5.00,30.0,2.3984,7.1953


## Inspecionar respostas

In [9]:
for _, row in df_r.head(5).iterrows():
    print(f"--- Q{int(row['id'])} ---")
    print(f"P: {row['pergunta']}")
    print(f"R: {row['resposta']}")
    print(f"   tokens={row['total_tokens']}  latencia={row['latencia_s']}s\n")

--- Q1 ---
P: Qual o período de inscrição para o concurso CVM?
R: As inscrições para o concurso da CVM estarão abertas do dia 30 de janeiro de 2024 até o dia 06 de março de 2024, com o horário de encerramento às 16h do último dia. O pagamento da taxa de inscrição deve ser realizado até o dia 07 de março de 2024.

ref: edital [4.1]
   tokens=7909  latencia=11.27s

--- Q2 ---
P: Qual o valor da taxa de inscrição do concurso CVM?
R: O valor da taxa de inscrição para o concurso da CVM é de R$ 145,00 (cento e quarenta e cinco reais).

ref: edital [4.1]
   tokens=6190  latencia=3.99s

--- Q3 ---
P: Em qual site o candidato deve fazer a inscrição do concurso CVM?
R: O candidato deve realizar a inscrição para o concurso da CVM acessando o site https://conhecimento.fgv.br/concursos/cvm24. As inscrições estarão abertas de 30 de janeiro de 2024 até 6 de março de 2024, e o pagamento da taxa de inscrição deve ser feito até o dia 7 de março de 2024. 

ref: edital [4.2]
   tokens=6764  latencia=3.68s